# Multiclass Image Classification with Convolutional Neural Networks (CNN)

**Course:** DXB-MSC AI-CS7003NU - Advanced AI Technologies  
**Topic:** Multiclass Image Classification using Deep Learning

---

## Learning Objectives
1. Understand how CNNs extract features from images
2. Build and train a multiclass classifier on the CIFAR-10 dataset
3. Monitor training progress with accuracy and loss curves
4. Evaluate model performance using confusion matrices and classification reports
5. Interpret predictions on unseen test images

## Dataset: CIFAR-10
- **60,000 color images** (32x32 pixels, 3 channels: RGB)
- **10 classes:** airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
- **50,000 training** images / **10,000 test** images

> **Note:** If `tensorflow` is not installed, run this command first:
> ```bash
> pip install tensorflow
> ```

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print("\nLibraries imported successfully!")

## 2. Load and Explore the CIFAR-10 Dataset

In [ ]:
# Load CIFAR-10 dataset (downloaded automatically on first run)
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training images: {x_train.shape}")   # (50000, 32, 32, 3)
print(f"Training labels: {y_train.shape}")   # (50000, 1)
print(f"Testing images:  {x_test.shape}")    # (10000, 32, 32, 3)
print(f"Testing labels:  {y_test.shape}")    # (10000, 1)
print(f"\nNumber of classes: {len(class_names)}")
print(f"Class names: {class_names}")

## 3. Visualize Sample Images from Each Class

In [ ]:
plt.figure(figsize=(12, 6))
for i in range(10):
    # Find first image of each class
    idx = np.where(y_train.flatten() == i)[0][0]
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[idx])
    plt.title(class_names[i], fontsize=12)
    plt.axis('off')
plt.suptitle('CIFAR-10: One Sample from Each Class', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

Neural networks perform better when input data is **normalized** (pixel values between 0 and 1).  
We also convert labels to **one-hot encoding** for the multiclass classification output.

In [ ]:
# Normalize pixel values to range [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Convert labels to one-hot encoded vectors
# Example: label '3' (cat) becomes [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print(f"Sample one-hot label for class {y_train[0][0]} ({class_names[y_train[0][0]]}):")
print(y_train_cat[0])
print(f"\nTraining data range: [{x_train.min():.2f}, {x_train.max():.2f}]")

## 5. Build the CNN Architecture

We design a **Convolutional Neural Network (CNN)** with the following structure:

| Layer Type | Purpose |
|------------|---------|
| **Conv2D** | Extracts features (edges, textures, shapes) using filters/kernels |
| **MaxPooling2D** | Reduces spatial dimensions, keeps important features |
| **Dropout** | Prevents overfitting by randomly deactivating neurons |
| **Flatten** | Converts 2D feature maps into a 1D vector |
| **Dense** | Fully connected layers for classification |
| **Softmax** | Outputs probabilities for each of the 10 classes |

> **Remember from class:** The number of hidden layers is determined experimentally (trial and error) by observing loss and accuracy curves!

In [ ]:
def build_cnn_model():
    """
    Builds a CNN for CIFAR-10 multiclass classification.
    Input:  32x32 RGB images
    Output: 10 class probabilities (softmax)
    """
    model = models.Sequential([
        # Block 1: Conv + Conv + Pool
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 2: Conv + Conv + Pool
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 3: Conv + Pool
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Flatten and Dense layers
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        
        # Output layer: 10 neurons (one per class) with softmax
        layers.Dense(10, activation='softmax')
    ])
    
    return model

# Build the model
model = build_cnn_model()

# Display model architecture
model.summary()

## 6. Compile the Model

We configure three essential components:
- **Optimizer:** `Adam` — adaptive learning rate optimizer (handles backpropagation)
- **Loss Function:** `categorical_crossentropy` — standard for multiclass classification
- **Metric:** `accuracy` — percentage of correctly classified images

In [ ]:
model.compile(
    optimizer='adam',                    # Optimizer: reduces loss via backpropagation
    loss='categorical_crossentropy',     # Loss: measures error for multiclass problems
    metrics=['accuracy']                 # Metric: tracks classification accuracy
)

print("Model compiled successfully!")
print("\nConfiguration:")
print("  Optimizer: Adam")
print("  Loss: Categorical Crossentropy")
print("  Metric: Accuracy")

## 7. Train the Model

We train for up to 50 epochs with two helpful callbacks:
- **EarlyStopping:** Stops training if validation loss stops improving (prevents overfitting)
- **ModelCheckpoint:** Saves the best model automatically

Watch how **loss reduces** and **accuracy improves** epoch by epoch — just like in class!

In [ ]:
# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',      # Watch validation loss
    patience=10,             # Wait 10 epochs before stopping
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'best_cifar10_model.keras',  # Save best model to disk
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("Training started...\n")

# Train the model
history = model.fit(
    x_train, y_train_cat,
    epochs=50,               # Maximum epochs (may stop early)
    batch_size=64,           # Process 64 images at a time
    validation_split=0.2,    # 20% of training data for validation
    callbacks=[early_stop, checkpoint],
    verbose=1                # Show progress bar each epoch
)

print("\nTraining complete!")

## 8. Plot Training Curves (Loss & Accuracy)

These curves are the **best way to verify backpropagation** and diagnose model health:
- **Loss curves:** Should decrease steadily. Large gap = overfitting.
- **Accuracy curves:** Training and validation should rise together.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss
axes[0].plot(history.history['loss'], label='Training Loss', color='blue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='orange')
axes[0].set_title('Model Loss Over Epochs', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (Categorical Crossentropy)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy', color='green')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', color='red')
axes[1].set_title('Model Accuracy Over Epochs', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final values
final_train_acc = history.history['accuracy'][-1] * 100
final_val_acc = history.history['val_accuracy'][-1] * 100
print(f"\nFinal Training Accuracy:   {final_train_acc:.2f}%")
print(f"Final Validation Accuracy: {final_val_acc:.2f}%")
print(f"Gap: {final_train_acc - final_val_acc:.2f}%")

## 9. Evaluate on the Test Set

The test set contains **10,000 images the model has never seen**.  
This gives us an unbiased estimate of real-world performance.

In [ ]:
# Evaluate on test data
test_loss, test_accuracy = model.evaluate(x_test, y_test_cat, verbose=0)

print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Generate predictions
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

print(f"\nPredictions generated for {len(y_pred)} test images.")

## 10. Confusion Matrix

A confusion matrix shows exactly which classes are being confused with each other.  
Diagonal values = correct predictions. Off-diagonal = misclassifications.

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - CIFAR-10 Test Set', fontsize=16)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Classification Report

Precision, Recall, and F1-Score for each class.
- **Precision:** Of all predicted as X, how many were actually X?
- **Recall:** Of all actual X, how many did we find?
- **F1-Score:** Harmonic mean of precision and recall.

In [ ]:
print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

## 12. Visualize Predictions on Sample Test Images

Let's look at 15 random test images and see how the model classifies them.  
We display the **predicted class** and the **confidence (probability)**.

- **Green label** = Correct prediction
- **Red label** = Wrong prediction

In [ ]:
# Select 15 random test images
num_images = 15
random_indices = np.random.choice(len(x_test), num_images, replace=False)

plt.figure(figsize=(15, 10))
for i, idx in enumerate(random_indices):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[idx])
    
    pred_class = y_pred[idx]
    true_class = y_true[idx]
    confidence = y_pred_probs[idx][pred_class] * 100
    
    color = 'green' if pred_class == true_class else 'red'
    title = f"Pred: {class_names[pred_class]}\nTrue: {class_names[true_class]}\nConf: {confidence:.1f}%"
    
    plt.title(title, color=color, fontsize=10)
    plt.axis('off')

plt.suptitle('Sample Test Predictions', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 13. Interpret Predictions with Probabilities

Just like the diabetes predictions in class (9% = no diabetes, 83% = diabetes),  
image classification outputs a **probability distribution** across all 10 classes.  
Let's examine the probability bar chart for a single image.

In [ ]:
# Pick a random test image
idx = np.random.randint(0, len(x_test))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Show the image
ax1.imshow(x_test[idx])
ax1.set_title(f"True Label: {class_names[y_true[idx]]}", fontsize=14)
ax1.axis('off')

# Show probability bars
probs = y_pred_probs[idx] * 100
colors = ['green' if i == y_pred[idx] else 'gray' for i in range(10)]
bars = ax2.bar(class_names, probs, color=colors)
ax2.set_ylim([0, 100])
ax2.set_ylabel('Probability (%)', fontsize=12)
ax2.set_title('Model Confidence per Class', fontsize=14)
ax2.tick_params(axis='x', rotation=45)

# Add percentage labels on bars
for bar, prob in zip(bars, probs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{prob:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Predicted class: {class_names[y_pred[idx]]} ({probs[y_pred[idx]]:.2f}% confidence)")
print(f"True class:      {class_names[y_true[idx]]}")

---

## Homework & Experiments

1. **Add More Layers:** Try adding another Conv2D block. Does accuracy improve?
2. **Data Augmentation:** Add `ImageDataGenerator` with rotations, flips, and zooms to reduce overfitting.
3. **Different Optimizers:** Compare `Adam`, `SGD`, and `RMSprop`. Which converges fastest?
4. **Learning Rate:** Adjust the learning rate in the optimizer. Too high = unstable; too low = slow.
5. **Batch Size:** Try `batch_size=32` vs `batch_size=128`. How does it affect training speed and accuracy?
6. **Transfer Learning:** Replace the custom CNN with a pre-trained model like ResNet50 or MobileNetV2.

> **Key Takeaway:** The gap between training accuracy and validation accuracy tells you if your model is overfitting. Use dropout, data augmentation, or early stopping to fix it!

## Model Architecture Diagram (Reference)

```
Input: 32x32x3 (RGB Image)
    |
    v
[Conv2D: 32 filters, 3x3] ---> [Conv2D: 32 filters] ---> [MaxPool 2x2] ---> [Dropout 25%]
    |
    v
[Conv2D: 64 filters, 3x3] ---> [Conv2D: 64 filters] ---> [MaxPool 2x2] ---> [Dropout 25%]
    |
    v
[Conv2D: 128 filters, 3x3] ---> [MaxPool 2x2] ---> [Dropout 25%]
    |
    v
[Flatten] ---> [Dense: 128 neurons] ---> [Dropout 50%]
    |
    v
[Dense: 10 neurons] ---> [Softmax]
    |
    v
Output: Probability for each of 10 classes
```